In [2]:
import pandas as pd
import numpy as np
import os

# --------------------------------------------------------------
# NSL-KDD raw files are expected at:
#   D:\Q-Harvest\data\raw\KDDTrain+.txt
#   D:\Q-Harvest\data\raw\KDDTest+.txt
# This notebook lives in D:\Q-Harvest\notebooks\, so we use
# relative paths (../data/raw/...) rather than hardcoded drive paths.
# --------------------------------------------------------------

RAW_DATA_DIR = "../data/raw"
TRAIN_PATH = os.path.join(RAW_DATA_DIR, "KDDTrain+.txt")
TEST_PATH = os.path.join(RAW_DATA_DIR, "KDDTest+.txt")

# Standard NSL-KDD column names (41 features + label + difficulty)
NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"
]

train_df_full = pd.read_csv(TRAIN_PATH, names=NSL_KDD_COLUMNS)
test_df_full = pd.read_csv(TEST_PATH, names=NSL_KDD_COLUMNS)

print(f"Loaded train: {train_df_full.shape[0]} rows")
print(f"Loaded test:  {test_df_full.shape[0]} rows")

# --------------------------------------------------------------
# The 11 features the quantum autoencoder uses
# --------------------------------------------------------------
FEATURE_COLUMNS = [
    "duration", "src_bytes", "dst_bytes", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate"
]

# X_train: train only on normal traffic (standard NSL-KDD anomaly-detection setup)
train_df_normal = train_df_full[train_df_full["label"] == "normal"].copy()
X_train = train_df_normal[FEATURE_COLUMNS].to_numpy(dtype=np.float64)

# X_test: all test traffic (normal + attacks), used later for evaluation
X_test = test_df_full[FEATURE_COLUMNS].to_numpy(dtype=np.float64)

# Build test_df with the binary_label column required by the threshold step
test_df = test_df_full.copy()
test_df["binary_label"] = np.where(test_df["label"] == "normal", "normal", "attack")

print(f"X_train (normal only): {X_train.shape}")
print(f"X_test  (all traffic): {X_test.shape}")
print(test_df["binary_label"].value_counts())

Loaded train: 125973 rows
Loaded test:  22544 rows
X_train (normal only): (67343, 11)
X_test  (all traffic): (22544, 11)
binary_label
attack    12833
normal     9711
Name: count, dtype: int64


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import pennylane as qml
from pennylane import qnn

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

import joblib

torch.manual_seed(42)
np.random.seed(42)

os.makedirs("../models", exist_ok=True)

print("PennyLane version:", qml.__version__)
print("Torch version:", torch.__version__)

PennyLane version: 0.45.1
Torch version: 2.13.0+cpu


In [4]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=4, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Rescale PCA outputs to [0, pi] for AngleEmbedding
pca_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_pca_angles = pca_scaler.fit_transform(X_train_pca)
X_test_pca_angles = pca_scaler.transform(X_test_pca)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_pca shape:    {X_train_pca.shape}")
print(f"Explained variance ratio (4 components): {pca.explained_variance_ratio_.sum():.4f}")

X_train_scaled shape: (67343, 11)
X_train_pca shape:    (67343, 4)
Explained variance ratio (4 components): 0.9485


In [5]:
N_QUBITS = 4
N_LAYERS = 3

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_LAYERS, N_QUBITS)}
quantum_torch_layer = qnn.TorchLayer(quantum_circuit, weight_shapes)

print("Quantum layer created.")
print("Weight shape:", weight_shapes)

Quantum layer created.
Weight shape: {'weights': (3, 4)}


In [6]:
class QuantumAutoencoder(nn.Module):
    def __init__(self, qnode, weight_shapes):
        super(QuantumAutoencoder, self).__init__()
        self.quantum_layer = qnn.TorchLayer(qnode, weight_shapes)
        # CRITICAL NAMING: must be exactly `classical_output`
        self.classical_output = nn.Linear(4, 11)

    def forward(self, x):
        q_out = self.quantum_layer(x)
        reconstruction = self.classical_output(q_out)
        return reconstruction


model = QuantumAutoencoder(quantum_circuit, weight_shapes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model moved to device: {device}")
print(model)

Model moved to device: cpu
QuantumAutoencoder(
  (quantum_layer): <Quantum Torch Layer: func=quantum_circuit>
  (classical_output): Linear(in_features=4, out_features=11, bias=True)
)


In [7]:
X_train_input_t = torch.FloatTensor(X_train_pca_angles).to(device)
X_train_target_t = torch.FloatTensor(X_train_scaled).to(device)

train_dataset = TensorDataset(X_train_input_t, X_train_target_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

N_EPOCHS = 30
loss_history = []

model.train()
for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        reconstruction = model(batch_x)
        loss = criterion(reconstruction, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_epoch_loss = epoch_loss / n_batches
    loss_history.append(avg_epoch_loss)
    print(f"Epoch [{epoch + 1:02d}/{N_EPOCHS}] - Avg MSE Loss: {avg_epoch_loss:.6f}")

print("\nTraining complete.")

Epoch [01/30] - Avg MSE Loss: 0.026038
Epoch [02/30] - Avg MSE Loss: 0.007059
Epoch [03/30] - Avg MSE Loss: 0.004321
Epoch [04/30] - Avg MSE Loss: 0.003429
Epoch [05/30] - Avg MSE Loss: 0.002967
Epoch [06/30] - Avg MSE Loss: 0.001890
Epoch [07/30] - Avg MSE Loss: 0.001268
Epoch [08/30] - Avg MSE Loss: 0.001073
Epoch [09/30] - Avg MSE Loss: 0.000977
Epoch [10/30] - Avg MSE Loss: 0.000935
Epoch [11/30] - Avg MSE Loss: 0.000915
Epoch [12/30] - Avg MSE Loss: 0.000899
Epoch [13/30] - Avg MSE Loss: 0.000889
Epoch [14/30] - Avg MSE Loss: 0.000880
Epoch [15/30] - Avg MSE Loss: 0.000873
Epoch [16/30] - Avg MSE Loss: 0.000867
Epoch [17/30] - Avg MSE Loss: 0.000862
Epoch [18/30] - Avg MSE Loss: 0.000856
Epoch [19/30] - Avg MSE Loss: 0.000852
Epoch [20/30] - Avg MSE Loss: 0.000857
Epoch [21/30] - Avg MSE Loss: 0.000847
Epoch [22/30] - Avg MSE Loss: 0.000846
Epoch [23/30] - Avg MSE Loss: 0.000844
Epoch [24/30] - Avg MSE Loss: 0.000843
Epoch [25/30] - Avg MSE Loss: 0.000841
Epoch [26/30] - Avg MSE L

In [8]:
model.eval()

X_test_input_t = torch.FloatTensor(X_test_pca_angles).to(device)
X_test_target_t = torch.FloatTensor(X_test_scaled).to(device)

with torch.no_grad():
    test_reconstructions = model(X_test_input_t)
    per_row_mse = torch.mean((test_reconstructions - X_test_target_t) ** 2, dim=1)
    per_row_mse_np = per_row_mse.cpu().numpy()

test_df_local = test_df.copy()
test_df_local["reconstruction_error"] = per_row_mse_np

normal_mask = test_df_local["binary_label"] == "normal"
normal_test_errors = test_df_local.loc[normal_mask, "reconstruction_error"].values

if len(normal_test_errors) == 0:
    raise ValueError(
        "No rows with binary_label == 'normal' found in test_df. "
        "Check that the label column and its values are correct."
    )

QUANTUM_THRESHOLD = np.percentile(normal_test_errors, 90) * 0.9

print(f"Number of normal test samples used for threshold: {len(normal_test_errors)}")
print(f"Mean normal reconstruction error:   {normal_test_errors.mean():.6f}")
print(f"90th percentile (raw):               {np.percentile(normal_test_errors, 90):.6f}")
print(f"QUANTUM_THRESHOLD (90th pct * 0.9):   {QUANTUM_THRESHOLD:.6f}")

predicted_anomaly = per_row_mse_np > QUANTUM_THRESHOLD
print(f"Flagged as anomaly: {predicted_anomaly.sum()} / {len(predicted_anomaly)} test samples")

Number of normal test samples used for threshold: 9711
Mean normal reconstruction error:   0.001248
90th percentile (raw):               0.000460
QUANTUM_THRESHOLD (90th pct * 0.9):   0.000414
Flagged as anomaly: 9166 / 22544 test samples


In [9]:
MODEL_PATH = "../models/quantum_autoencoder.pth"
SCALER_PATH = "../models/scaler.pkl"
PCA_PATH = "../models/pca_transformer.pkl"
PCA_ANGLE_SCALER_PATH = "../models/pca_angle_scaler.pkl"
THRESHOLD_PATH = "../models/quantum_threshold.pkl"

torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(pca, PCA_PATH)
joblib.dump(pca_scaler, PCA_ANGLE_SCALER_PATH)
joblib.dump(QUANTUM_THRESHOLD, THRESHOLD_PATH)

print("Artifacts saved successfully:")
print(f"  Model:              {MODEL_PATH}")
print(f"  Scaler:              {SCALER_PATH}")
print(f"  PCA transformer:     {PCA_PATH}")
print(f"  PCA angle scaler:    {PCA_ANGLE_SCALER_PATH}")
print(f"  Quantum threshold:   {THRESHOLD_PATH}  (value: {QUANTUM_THRESHOLD:.6f})")

Artifacts saved successfully:
  Model:              ../models/quantum_autoencoder.pth
  Scaler:              ../models/scaler.pkl
  PCA transformer:     ../models/pca_transformer.pkl
  PCA angle scaler:    ../models/pca_angle_scaler.pkl
  Quantum threshold:   ../models/quantum_threshold.pkl  (value: 0.000414)
